# 개별종목 조합B — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합B 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합B의 피처 값만 지정합니다.
import json

COMBINATION = 'B'
FEATURE_COLUMNS = (
    'ret_5',
    'ret_20',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'dist_high_20',
    'dist_high_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합B 피처: ('ret_5', 'ret_20', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'dist_high_20', 'dist_high_60')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4710,0.5012,-0.0303,0.3566,0.3704,0.0746,0.3906,0.1713,0.2787
1,2,balanced,980,20150123,20150421,0.3799,0.3978,-0.0180,0.3549,0.3608,0.0480,0.3739,0.2627,0.3241
2,3,balanced,1210,20151228,20160328,0.3625,0.3762,-0.0137,0.3589,0.3590,0.0384,0.3604,0.3124,0.3430
3,4,balanced,1439,20161202,20170228,0.4540,0.4617,-0.0077,0.3949,0.3987,0.1121,0.4098,0.2401,0.3371
4,5,balanced,1669,20171113,20180207,0.4007,0.3901,0.0107,0.3741,0.3812,0.0761,0.3913,0.2760,0.3412
5,6,balanced,1899,20181024,20190118,0.4348,0.3725,0.0623,0.4319,0.4322,0.1505,0.4403,0.3877,0.4170
6,7,balanced,2129,20190930,20191224,0.4378,0.4781,-0.0403,0.3520,0.3667,0.0636,0.3892,0.2471,0.3271
7,8,balanced,2359,20200902,20201130,0.4075,0.3476,0.0599,0.4068,0.4121,0.1179,0.3990,0.4505,0.4206
8,9,balanced,2589,20210806,20211105,0.3578,0.3914,-0.0336,0.3412,0.3634,0.0349,0.3620,0.1865,0.2705
9,10,balanced,2818,20220714,20221012,0.3765,0.3454,0.0311,0.3729,0.3832,0.0739,0.3816,0.2594,0.3263


,OOS 폴드 평균
accuracy,0.4038
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0070
macro_f1,0.3747
balanced_accuracy,0.3818
mcc,0.0772
pr_auc_macro_ovr,0.3899
down_recall,0.2959
core_harmonic_mean,0.3453


재실행 명령: python scripts/run_stock_model_experiment.py
